#### Loading packages

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf


#### Setting root directory path

In [ ]:
ROOT = r'C:\Users\PC_DS_ECON_5\Desktop\data-analytics-python'


#### Data source used: NLSY97

The NLSY97 is a `longitudinal` (i.e. `panel`) survey of individuals born between 1980 and 1984 (country: U.S.). The first round of data collection took place in 1997 and was followed by additional survey rounds every one or two years.

**Selected data**

For the analysis below, let us use data on the current job (earnings, tenure, hours, occupation, industry) in 2023 and demographic, family background, as well as educational variables.


#### Loading data

Reminder: We need the `pyarrow` package to import a `.parquet` datafile.

In [ ]:
nlsy97_current_jobs_2023 = pd.read_parquet(ROOT + '/data/nlsy97_current_jobs_2023.parquet')

#### How much of the gender pay gap can be explained by differences in observable characteristics such as education, occupation, industry, and tenure?


In a previous session, we saw that there is a gender pay gap: on average, women earn a lower hourly wage than men.

We measured the gender pay gap using the following symmetric formula:

$$
GPG = \frac{\mathbb{E}[hpay \mid gender=\text{Female}] - \mathbb{E}[hpay \mid gender=\text{Male}]}{\max\{\mathbb{E}[hpay \mid gender=\text{Female}], \mathbb{E}[hpay \mid gender=\text{Male}] \}}
$$

Let us compute the exact value of the gender pay gap in our sample:

In [ ]:
df = nlsy97_current_jobs_2023.copy()
df = df.loc[df['sample'].eq('Cross-sectional') & df['hpay'].gt(0),:].copy()
female_mean_hpay = df.loc[df['gender'].eq('Female'), 'hpay'].mean()
male_mean_hpay = df.loc[df['gender'].eq('Male'), 'hpay'].mean()

gpg = (female_mean_hpay - male_mean_hpay)/(max(female_mean_hpay, male_mean_hpay))
gpg

We obtain the same estimated gender pay gap as before:

$$
\widehat{GPG} = \frac{\overline{hpay}_{\mathrm{female}}-\overline{hpay}_{\mathrm{male}}}
{\max\left\{\overline{hpay}_{\mathrm{female}},\overline{hpay}_{\mathrm{male}}\right\}} \approx -0.17 = -17\%
$$

Women earn, on average, about 17% less per hour than men according to this measure.

We can also measure the gender pay gap using a simple linear regression.

To do so, let us first define a dummy variable `female`:

$$
female = 
\begin{cases}
1 & \text{ if } gender = \text{Female}\\
0 & \text{ otherwise }
\end{cases}
$$


In [ ]:
df = nlsy97_current_jobs_2023.copy()

df['female'] = np.where(df['gender'].notna(), 
                        np.where(df['gender'].eq('Female'),1,0),
                        pd.NA)
df = df.convert_dtypes()
nlsy97_current_jobs_2023 = df.copy()  

And then let us regress `hpay` on the dummy variable `female`:

$$
hpay = \beta_0 + \beta_1 \cdot female + \varepsilon 
$$

If $\beta_0$ and $\beta_1$ are the OLS coefficients and $\varepsilon$ is the approximation error, then:
 - the mean hourly pay among men is the intercept: $\mathbb{E}[hpay \mid female=0] = \beta_0 $
 - the mean hourly pay among women is the sum of the intercept and the slope coefficient: $\mathbb{E}[hpay \mid female=1] = \beta_0 + \beta_1 $

The reason why we can directly interpret the coefficients in terms of conditional means is that there are **exactly as many parameters ($\beta_0$, $\beta_1$) as conditional means** (`female=1`, `female=0`). In this special case, the regression directly estimates the conditional mean function, that is, the conditional mean of the outcome for each value of the regressor. It is no longer just the best linear approximation to the conditional mean function, because **the best linear approximation and the conditional mean function coincide**. 

Therefore, the slope coefficient is the difference in mean hourly pay between women and men:

$$
\beta_1 = \mathbb{E}[hpay \mid female=1] - \mathbb{E}[hpay \mid female=0]
$$

Our formula above can thus be written as:

$$
GPG = \frac{\beta_1}{\max\{\beta_0 + \beta_1, \beta_0\}}
$$

In [ ]:
df = nlsy97_current_jobs_2023.copy()
df = df.loc[df['sample'].eq('Cross-sectional') & df['hpay'].gt(0), :].reset_index(drop=True).copy()

outcome = 'hpay'
regressor = 'female'
formula = outcome + ' ~ ' + regressor 
df = df[[outcome, regressor]].dropna().reset_index().copy()

results = smf.ols(formula=formula, data=df).fit()

results.params

For our sample, we obtain:

$$
\widehat{hpay}_i = 41.7 - 7.15 \cdot female_i
$$

Interpretation:
 - $\widehat{\beta}_0 = 41.7$: mean hourly pay among men: $\overline{hpay}_{\mathrm{male}}$
 - $\widehat{\beta}_1 = -7.15$: difference in mean hourly pay between women and men: $\overline{hpay}_{\mathrm{female}} - \overline{hpay}_{\mathrm{male}}$

And we get our sample plug-in estimate:

$$
\widehat{GPG} = \frac{\widehat{\beta}_1}{\max\{\widehat{\beta}_0 + \widehat{\beta}_1, \widehat{\beta}_0\}} = \frac{-7.15}{\max\{41.7 - 7.15, 41.7\}} \approx -0.17 = -17\% 
$$


We could get a direct proportional interpretation by regressing log hourly pay `log_hpay` instead of hourly pay:

$$
log\_hpay = \beta_0 + \beta_1 \cdot female + \varepsilon
$$

If $\beta_0$ and $\beta_1$ are the OLS coefficients and $\varepsilon$ is the approximation error, then:
 - the mean log hourly pay among men is the intercept: $\mathbb{E}[log\_hpay \mid female=0] = \beta_0 $
 - the difference in mean log hourly pay among women is the sum of the intercept and the slope coefficient: $\mathbb{E}[log\_hpay \mid female=1] = \beta_0 + \beta_1 $


Therefore, the slope coefficient is the difference in mean hourly pay between women and men:

$$
\beta_1 = \mathbb{E}[log\_hpay \mid female=1] - \mathbb{E}[log\_hpay \mid female=0]
$$


Defining the log hourly pay variable `log_hpay`.  

In [ ]:
df = nlsy97_current_jobs_2023.copy()
df['log_hpay'] = pd.NA
valid = df['hpay'].gt(0)
df.loc[valid, 'log_hpay'] = np.log(df.loc[valid, 'hpay'])
df = df.convert_dtypes()
nlsy97_current_jobs_2023 = df.copy()

Running the regression:

In [ ]:
df = nlsy97_current_jobs_2023.copy()
df = df.loc[df['sample'].eq('Cross-sectional'), :].reset_index(drop=True).copy()

outcome = 'log_hpay'
regressor = 'female'
formula = outcome + ' ~ ' + regressor 
df = df[[outcome, regressor]].dropna().reset_index().copy()

results = smf.ols(formula=formula, data=df).fit()

results.params

For our sample, we obtain:

$$
\widehat{\log(hpay)}_i = 3.48 - 0.186 \cdot female_i
$$

Conditional means interpretation:

- $\widehat{\beta}_0 = 3.48$: mean log hourly pay among men:
  $\overline{\log(hpay)}_{\mathrm{male}}$
- $\widehat{\beta}_1 = -0.186$: difference in mean log hourly pay between women and men:
  $\overline{\log(hpay)}_{\mathrm{female}} - \overline{\log(hpay)}_{\mathrm{male}}$

Predicted-value interpretation:

$$
\widehat{\log(hpay)}_i
=
3.48
-
0.186 \cdot female_i
$$

Exponentiating both sides gives the predicted hourly pay:

$$
\widehat{hpay}_i
=
e^{3.48 - 0.186 \cdot female_i}
$$

Taking the proportional difference between the predicted hourly pay of women and men:

$$
\frac{\widehat{hpay}_{\mathrm{female}}
-
\widehat{hpay}_{\mathrm{male}}}
{\widehat{hpay}_{\mathrm{male}}}
=
\frac{
e^{3.48-0.186\cdot1}
-
e^{3.48-0.186\cdot0}
}
{
e^{3.48-0.186\cdot0}
}
$$

Using the properties of exponentiation:

$$
\frac{\widehat{hpay}_{\mathrm{female}}
-
\widehat{hpay}_{\mathrm{male}}}
{\widehat{hpay}_{\mathrm{male}}}
=
e^{-0.186}-1
=
-0.1697
\approx
-17\%.
$$

Because the exponential function is non-linear, the proportional difference between exponentiated conditional mean log hourly pay is generally not identical to the proportional difference between conditional mean hourly pay. In this example, however, the numerical difference is very small.

This is why

- the proportional difference in exponentiated conditional mean log hourly pay:

$$
\frac{
e^{\overline{\log(hpay)}_{\mathrm{female}}}
-
e^{\overline{\log(hpay)}_{\mathrm{male}}}
}
{
e^{\overline{\log(hpay)}_{\mathrm{male}}}
}
=
-0.1697,
$$

and

- the proportional difference in conditional mean hourly pay:

$$
\frac{
\overline{hpay}_{\mathrm{female}}
-
\overline{hpay}_{\mathrm{male}}
}
{
\overline{hpay}_{\mathrm{male}}
}
=
-0.1712
$$

are often treated as approximately equivalent in practice.

##### From unconditional to conditional gender pay gaps

Up to this point, the object of interest has been the **unconditional** difference in mean hourly pay between men and women.

This unconditional gap may itself be of interest for many reasons (e.g. savings, pensions, bargaining power within the household).

However, we may also wish to understand **how much of this gap can be attributed to observable characteristics**.

Simple examples are that men and women are not equally likely

- to work in the same occupations;
- to have the same educational attainment;
- etc.

In other words, **composition effects** may contribute to the unconditional gender pay gap.

For example, we may be interested in the gender pay gap **while holding occupation constant**, or **while holding education constant**.

This is precisely what **multiple linear regression** is useful for.

Let us start by controlling for education.

Education is measured by the categorical variable `educ_cat`, which has four categories.

We include three education dummy variables, using the fourth category `Less than High School` as the reference category:

$$
\begin{array}{rcl}
log\_hpay &=&
\beta_0
+ \beta_1 \cdot female
+ \delta_{HS} \cdot \mathbb{1}[educ\_cat=\text{High School}]\ + \\ \\
&& \ \
+\ \delta_{SC} \cdot \mathbb{1}[educ\_cat=\text{Some College}]
+ \delta_{BP} \cdot \mathbb{1}[educ\_cat=\text{Bachelor Plus}]
+ \varepsilon.
\end{array}
$$

Here, we can no longer interpret the coefficients as conditional means. The reason is that the number of parameters (5 coefficients: $\beta_0$, $\beta_1$, $\delta_{HS}$, $\delta_{SC}$, $\delta_{BP}$) is smaller than the number of conditional means to be estimated (2 genders × 4 education categories = 8 conditional means).

The regression therefore imposes restrictions on the conditional mean function.

Unlike the previous regression, this specification is therefore only the best linear approximation to the conditional mean function.

The variable `educ_cat` is already stored as an ordered categorical variable:

In [ ]:
nlsy97_current_jobs_2023['educ_cat'].dtype

As a result, it can be included directly in the statsmodels formula interface and the first category in the specified category ordering is automatically used as the reference category.

In [ ]:
df = nlsy97_current_jobs_2023.copy()

df['educ_cat'].dtype


outcome = 'log_hpay'
regressors = ['female', 'educ_cat']
formula = 'log_hpay ~ female + educ_cat'

df = df[[outcome] + regressors].dropna().reset_index().copy()

results = smf.ols(formula=formula, data=df).fit()

results.params

In our sample, the estimated equation is:

$$
\begin{array}{rcl}
\widehat{log\_hpay} &=&
3.073
- 0.244 \cdot female
+ 0.177 \cdot \mathbb{1}[educ\_cat=\text{High School}] \ + \\
\\
&& \ \ 
+\ 0.332 \cdot \mathbb{1}[educ\_cat=\text{Some College}]
+ 0.773 \cdot \mathbb{1}[educ\_cat=\text{Bachelor Plus}]
\end{array}
$$

Using the predicted value interpretation together with the log-level interpretation:

- $\widehat{\beta}_0 = 3.073$: predicted log hourly pay for men ($female=0$) without a high school degree (the reference category for educational attainment)
- $\widehat{\beta}_1 = -0.244$: holding education constant, women are predicted to earn 21.7% less per hour than men on average ($e^{-0.244}-1 \approx -0.217 = -21.7\%$)
- $\widehat{\delta}_{HS} = 0.177$: holding gender constant, individuals whose highest level of education is a high school degree are predicted to earn 19.4% more per hour on average than individuals without a high school degree ($e^{0.177}-1 \approx 0.194 = 19.4\%$)
- $\widehat{\delta}_{SC} = 0.332$: holding gender constant, individuals who have attended college but do not have a Bachelor's degree are predicted to earn 39.4% more per hour on average than individuals without a high school degree ($e^{0.332}-1 \approx 0.394 = 39.4\%$)
- $\widehat{\delta}_{BP} = 0.773$: holding gender constant, individuals who have a Bachelor's degree or higher are predicted to earn 117% more per hour on average than individuals without a high school degree ($e^{0.773}-1 \approx 1.17 = 117\%$)

In [ ]:
for coef in results.params.index.drop(['Intercept']):
    print(coef, np.exp(results.params[coef]) - 1)

Overall, the gender pay gap seems to be stronger when controlling for education (21% versus 17%). 

In the previous regression, there were however restrictions: returns to education were the same within each gender and gender pay gaps were the same within each education category. 

Are there higher returns to education for males? What part of the gender pay gap is composition?

Let us run a fully flexible specification, estimating returns to educational attainment separately for each gender.

The regression equation now contains a full interaction of both categorical variables. We should have now eight parameters:

$$
\begin{array}{rcl}
log\_hpay &=&
\beta^{\mathrm{male}}_{0}
+ \gamma^{\mathrm{female}}_{0} \cdot female
+ \delta^{\mathrm{male}}_{HS} \cdot \mathbb{1}[educ\_cat=\text{High School}]\ + \gamma^{\mathrm{female}}_{HS} \cdot female \cdot \mathbb{1}[educ\_cat=\text{High School}]  \ + \\ \\
&& \ \
+\ \delta^{\mathrm{male}}_{SC} \cdot \mathbb{1}[educ\_cat=\text{Some College}] + \gamma^{\mathrm{female}}_{SC} \cdot female \cdot \mathbb{1}[educ\_cat=\text{Some College}]  \ + \\ \\
&& \ \
+ \delta^{\mathrm{male}}_{BP} \cdot \mathbb{1}[educ\_cat=\text{Bachelor Plus}] + \gamma^{\mathrm{female}}_{BP} \cdot female \cdot \mathbb{1}[educ\_cat=\text{Bachelor Plus}]
+ \varepsilon.
\end{array}
$$

Because we are not just approximating but exactly matching the conditional mean function $\mathbb{E}[log\_hpay \mid female, educ\_cat]$, we have a perfect correspondence between the coefficients and the conditional means:

$$
\begin{array}{rcl}
\mathbb{E}[log\_hpay \mid female=0, educ\_cat=\text{Less than High School}]  &=& \beta^{\mathrm{male}}_{0}  \\ \\
 \mathbb{E}[log\_hpay \mid female=1, educ\_cat=\text{Less than High School}] &=& \beta^{\mathrm{male}}_{0} + \gamma^{\mathrm{female}}_{0} \\ \\
\mathbb{E}[log\_hpay \mid female=0, educ\_cat=\text{High School}]  &=& \beta^{\mathrm{male}}_{0} + \delta^{\mathrm{male}}_{HS} \\ \\
\mathbb{E}[log\_hpay \mid female=1, educ\_cat=\text{High School}] &=& \beta^{\mathrm{male}}_{0} + \gamma^{\mathrm{female}}_{0} + \delta^{\mathrm{male}}_{HS} + \gamma^{\mathrm{female}}_{HS} \\ \\
\mathbb{E}[log\_hpay \mid female=0, educ\_cat=\text{Some College}] &=& \beta^{\mathrm{male}}_{0} + \delta^{\mathrm{male}}_{SC}  \\ \\
\mathbb{E}[log\_hpay \mid female=1, educ\_cat=\text{Some College}] &=& \beta^{\mathrm{male}}_{0} + \gamma^{\mathrm{female}}_{0} + \delta^{\mathrm{male}}_{SC} + \gamma^{\mathrm{female}}_{SC}  \\ \\
\mathbb{E}[log\_hpay \mid female=0, educ\_cat=\text{Bachelor Plus}] &=& \beta^{\mathrm{male}}_{0} + \delta^{\mathrm{male}}_{BP} \\ \\
\mathbb{E}[log\_hpay \mid female=1, educ\_cat=\text{Bachelor Plus}] &=& \beta^{\mathrm{male}}_{0} + \gamma^{\mathrm{female}}_{0} + \delta^{\mathrm{male}}_{BP} + \gamma^{\mathrm{female}}_{BP}
\end{array}
$$


Let us run the regression above and let us also count the shares of each educational category within each gender: 

In [ ]:
df = nlsy97_current_jobs_2023.copy()

df['educ_cat'].dtype


outcome = 'log_hpay'
regressors = ['female', 'educ_cat']
formula = 'log_hpay ~ female * educ_cat'

df = df[[outcome] + regressors].dropna().reset_index().copy()
results = smf.ols(formula=formula, data=df).fit()
results.params




Let us construct the returns by group and let us also count the share of individuals by education group within each gender:
 - The returns for females can be obtained as follows: 
    - $\delta^{\mathrm{female}}_{HS} = \delta^{\mathrm{male}}_{HS} + \gamma^{\mathrm{female}}_{HS}$
    - $\delta^{\mathrm{female}}_{SC} = \delta^{\mathrm{male}}_{SC} + \gamma^{\mathrm{female}}_{SC}$
    - $\delta^{\mathrm{female}}_{BP} = \delta^{\mathrm{male}}_{BP} + \gamma^{\mathrm{female}}_{BP}$


In [ ]:
educ_cat_reference = 'Less than High School'

coeff = pd.DataFrame({'value': results.params}).reset_index().rename(columns={'index': 'name'})
coeff['educ_cat'] = coeff['name'].str.split('.', expand=True)[1]
coeff['educ_cat'] = coeff['educ_cat'].str.removesuffix(']')
coeff['female'] = coeff['name'].str.match('female').astype(int)
coeff = coeff.convert_dtypes()
coeff.loc[coeff['educ_cat'].isna(), 'educ_cat'] = educ_cat_reference
coeff = coeff.drop(columns='name').copy()

coeff_wide = coeff.pivot_table(columns='female', values='value', index='educ_cat').reset_index().sort_values('educ_cat').reset_index(drop=True).copy()
# coeff_wide = coeff_wide.rename(columns={0: 'male', 1: 'female'})

coeff_wide[1] = coeff_wide[0] + coeff_wide[1]
coeff_wide.loc[coeff_wide['educ_cat'].eq(educ_cat_reference), 0] = 0
coeff_wide.loc[coeff_wide['educ_cat'].eq(educ_cat_reference), 1] = 0
returns = coeff_wide.melt(id_vars='educ_cat', var_name='female', value_name='value').sort_values(['female', 'educ_cat']).reset_index(drop=True).copy()

returns = returns.astype({
                            'educ_cat': df['educ_cat'].dtype,
                            'female': df['female'].dtype,
                        }).sort_values(['female', 'educ_cat']).reset_index(drop=True).copy()
returns = returns.rename(columns={'value': 'relative'})




educ_cat_shares = df.groupby('female').value_counts(normalize=True, subset=['educ_cat']).reset_index().copy()

reference_cat_vals = coeff.loc[coeff['educ_cat'].eq(educ_cat_reference), ['female', 'value']].copy()
reference_cat_vals.loc[coeff['female'].eq(1), ['value']] = reference_cat_vals['value'].sum()
reference_cat_vals = reference_cat_vals.rename(columns={'value': 'base'})

returns = pd.merge(left=returns,
         right=reference_cat_vals,
         on=['female'],
         how='left')

returns['absolute'] = returns['base'] + returns['relative']

returns

educ_cat_shares


In [ ]:
plot_df = returns.copy()

gender_labels = {0: 'Men', 1: 'Women'}
plot_df['gender'] = plot_df['female'].map(gender_labels)

education_order = df['educ_cat'].cat.categories
plot_df['educ_cat'] = plot_df['educ_cat'].astype(df['educ_cat'].dtype)

x_base = np.arange(len(education_order))
dodge = 0.18
bar_width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))

for female_value, offset in [(0, -dodge), (1, dodge)]:
    tmp = (
        plot_df
        .loc[plot_df['female'].eq(female_value)]
        .sort_values('educ_cat')
    )

    x = x_base + offset
    label = gender_labels[female_value]

    ax.bar(
        x=x,
        height=tmp['base'],
        width=bar_width,
        label=f'{label}: base',
        alpha=0.4,
    )

    ax.bar(
        x=x,
        height=tmp['relative'],
        bottom=tmp['base'],
        width=bar_width,
        label=f'{label}: education return',
        alpha=0.8,
    )

    for xi, y in zip(x, tmp['absolute']):
        ax.text(
            xi,
            y + 0.03,
            f'{y:.2f}',
            ha='center',
            va='bottom',
            fontsize=9,
        )

ax.set_xticks(x_base)
ax.set_xticklabels(education_order, rotation=20, ha='right')


ax.set_ylabel('Predicted log hourly pay')
ax.set_xlabel('Educational attainment')
ax.set_title('Predicted log hourly pay by gender and education')

ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
shares = (
    educ_cat_shares
    .sort_values(['female', 'educ_cat'])
    .copy()
)

education_order = df['educ_cat'].cat.categories
shares['educ_cat'] = shares['educ_cat'].astype(df['educ_cat'].dtype)

y = np.arange(len(education_order))
dodge = 0.18

fig, ax = plt.subplots(figsize=(6, 4))

for female, offset, label in [
    (0, -dodge, 'Men'),
    (1,  dodge, 'Women')
]:
    tmp = shares.loc[shares['female'].eq(female)].sort_values('educ_cat')

    ax.barh(
        y + offset,
        tmp['proportion'],      # oder 'prop', je nach Name
        height=0.32,
        label=label,
    )

ax.set_yticks(y)
ax.set_yticklabels(education_order)
ax.set_xlabel('Share')
ax.set_title('Educational attainment by gender')
ax.legend()

plt.tight_layout()

The overall picture is mixed. At the intermediate educational levels, men are slightly overrepresented and the gender pay gaps are somewhat larger. At the two extremes of the education distribution, women are slightly overrepresented and the gender pay gaps are somewhat smaller.

The contribution of educational attainment to the overall gender pay gap is therefore not immediately clear from the figure alone.

One thing appears to be clear: educational attainment cannot fully explain the gender pay gap, because substantial gender pay gaps remain within every education category.

Let us now investigate whether occupational sorting provides a clearer explanation. We start with the same regression as before, replacing education categories with occupation categories.

In [ ]:
df = nlsy97_current_jobs_2023.copy()
df = df.loc[df['sample'].eq('Cross-sectional'),:].copy()
occ_cat_shares = df.groupby('female').value_counts(normalize=True, subset=['occupation_group']).reset_index().copy()
occ_cat_n = df.groupby('female').value_counts(subset=['occupation_group']).reset_index().copy()

In [ ]:
df = nlsy97_current_jobs_2023.copy()
df = df.loc[df['sample'].eq('Cross-sectional'),:].copy()
occ_to_remove = occ_cat_n.loc[occ_cat_n['count'].lt(5),:].loc[:,'occupation_group'].drop_duplicates().to_list()
df = df.loc[~df['occupation_group'].isin(occ_to_remove),:].copy()

outcome = 'log_hpay'
regressors = ['female', 'occupation_group']
formula = 'log_hpay ~ female + occupation_group'

df = df[[outcome] + regressors].dropna().reset_index().copy()

results = smf.ols(formula=formula, data=df).fit()


results.params['female']

In [ ]:
np.exp(results.params['female']) - 1

There is still an 17.2% gender pay gap when holding occupation constant.

<!-- #### Oaxaca-Blinder decomposition

The question is explicitly to decompose the gender pay gap into two components:
 - Component due to the allocation of women versus men across occupations.
 - Differential returns to occupations for women versus men.

It is tightly linked to the educational attainment example.

The difference consists in that there is no way to unambiguously rank occupations and there are many more categories.

The objective is to get a summary measure.

To do so, let us run three regressions:
 - Regression with female dummy and occupation dummys. R-Pooled
 - Regression for women with occupation dummies. R-Women
 - Regression for men with occupation dummies. R-Men

A version of the Oaxaca-Blinder decomposition corresponds to:

$$
\begin{array}{rcl}
(\text{$\overline{log\_hpay}$ for Women}) - (\text{$\overline{log\_hpay}$ for Men})  &=& \underbrace{[(\text{$\overline{\widehat{log\_hpay}}$ for Women using R-Pooled}) - (\text{$\overline{\widehat{log\_hpay}}$ for Men Using R-Pooled})]}_{\text{composition effect}} \ + \\ \\
&& \ \ +\ \underbrace{[(\text{$\overline{\widehat{log\_hpay}}$ for Women using R-Women}) - (\text{$\overline{\widehat{log\_hpay}}$ for Women using R-Pooled})] + [(\text{$\overline{\widehat{log\_hpay}}$ for Men Using R-Pooled}) - (\text{$\overline{\widehat{log\_hpay}}$ for Men Using R-Men})]}_{\text{unexplained by composition effect}}
\end{array}
$$
 -->



#### Oaxaca-Blinder decomposition

The objective is to decompose the overall gender pay gap into two components:

- the component due to the different allocation of women and men across occupations (**composition effect**);
- the component due to different returns to occupations for women and men (**returns effect**).

The idea is closely related to the educational attainment example.

The main difference is that occupations cannot be naturally ordered and there are many more occupation categories.

Rather than inspecting the gender pay gap within every occupation separately, we would like to summarize the contribution of occupational sorting with a single measure.

To do so, let us estimate three regression models:

- **Pooled regression:** regression with a female dummy and occupation dummy variables.
- **Women's regression:** regression estimated only for women, using occupation dummy variables.
- **Men's regression:** regression estimated only for men, using occupation dummy variables.

To simplify the notation, let

- $\widehat{\mu}^{P}_{W}$ denote the mean predicted log hourly pay for women using the pooled regression;
- $\widehat{\mu}^{P}_{M}$ denote the mean predicted log hourly pay for men using the pooled regression;
- $\widehat{\mu}^{W}_{W}$ denote the mean predicted log hourly pay for women using the women's regression;
- $\widehat{\mu}^{M}_{M}$ denote the mean predicted log hourly pay for men using the men's regression.

One version of the Oaxaca-Blinder decomposition is then given by

$$
\begin{aligned}
(\overline{log\_hpay}_{W})
-
(\overline{log\_hpay}_{M})
&=
\underbrace{
\left(
\widehat{\mu}^{P}_{W}
-
\widehat{\mu}^{P}_{M}
\right)
}_{\text{composition effect}}
+
\underbrace{
\left(
\widehat{\mu}^{W}_{W}
-
\widehat{\mu}^{P}_{W}
\right)
+
\left(
\widehat{\mu}^{P}_{M}
-
\widehat{\mu}^{M}_{M}
\right)
}_{\text{returns effect (unexplained by composition)}}.
\end{aligned}
$$
